# Análise de dados TCP-CII

In [13]:
import pandas as pd

In [14]:
df = pd.read_csv('./T CELL/DENV 3 - T Cell Prediction - Class II.csv')
df

,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhciipan_el core,netmhciipan_el score,netmhciipan_el percentile
0,1,KNGSWKLEKASLIEVK,206,221,16,HLA-DRB1*01:01,246,0.02,WKLEKASLI,0.983079,0.02
1,1,QYKFQADSPKRLA,31,43,13,HLA-DRB3*01:01,7,0.05,FQADSPKRL,0.927520,0.05
2,1,KNGSWKLEKASLIEVKT,206,222,17,HLA-DRB1*01:01,314,0.06,WKLEKASLI,0.972226,0.06
3,1,QYKFQADSPKRLAT,31,44,14,HLA-DRB3*01:01,75,0.06,FQADSPKRL,0.921728,0.06
4,1,KNGSWKLEKASLIEVKTC,206,223,18,HLA-DRB1*01:01,382,0.07,WKLEKASLI,0.950728,0.07
...,...,...,...,...,...,...,...,...,...,...,...
16411,1,KLIHEWCCRSCTLPPLRYMGE,306,326,21,HLA-DRB3*02:02,603,100.00,CTLPPLRYM,0.000019,100.00
16412,1,KLIHEWCCRSCTLPPLRYMG,306,325,20,HLA-DRB1*04:01,536,100.00,CRSCTLPPL,0.000018,100.00
16413,1,KLIHEWCCRSCTLPPLRYMGE,306,326,21,HLA-DRB1*04:01,603,100.00,CRSCTLPPL,0.000015,100.00
16414,1,TTVSGKLIHEWCC,301,313,13,HLA-DRB3*02:02,61,100.00,VSGKLIHEW,0.000013,100.00


## Selecionando Epítopos com median binding percentile menor que 5.

In [15]:
df_mbp_m5 = df[df['median binding percentile'] < 5].copy()
df_mbp_m5

,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhciipan_el core,netmhciipan_el score,netmhciipan_el percentile
0,1,KNGSWKLEKASLIEVK,206,221,16,HLA-DRB1*01:01,246,0.02,WKLEKASLI,0.983079,0.02
1,1,QYKFQADSPKRLA,31,43,13,HLA-DRB3*01:01,7,0.05,FQADSPKRL,0.927520,0.05
2,1,KNGSWKLEKASLIEVKT,206,222,17,HLA-DRB1*01:01,314,0.06,WKLEKASLI,0.972226,0.06
3,1,QYKFQADSPKRLAT,31,44,14,HLA-DRB3*01:01,75,0.06,FQADSPKRL,0.921728,0.06
4,1,KNGSWKLEKASLIEVKTC,206,223,18,HLA-DRB1*01:01,382,0.07,WKLEKASLI,0.950728,0.07
...,...,...,...,...,...,...,...,...,...,...,...
527,1,VTNEVHTWTEQYKFQAD,21,37,17,HLA-DQA1*04:01/DQB1*04:02,277,4.90,VHTWTEQYK,0.115821,4.90
528,1,KTWGKAKIVTAETQNSSF,116,133,18,HLA-DQA1*04:01/DQB1*04:02,364,4.90,IVTAETQNS,0.094425,4.90
529,1,KNGSWKLEKASLIEV,206,220,15,HLA-DPA1*02:01/DPB1*14:01,178,4.90,WKLEKASLI,0.080230,4.90
530,1,KNGSWKLEKASLIE,206,219,14,HLA-DPA1*02:01/DPB1*14:01,110,4.90,WKLEKASLI,0.069204,4.90


## Agrupando por pepitideos e agregando colunas pertinentes.

In [16]:
epitopos_repetidos = (
    df_mbp_m5
    .groupby('peptide', as_index=False)
    .agg(
        start=("start", "first"),
        end=("end", "first"),
        qte_de_alelos=("allele", "nunique"),
        median_binding_percentile=(
            "median binding percentile",
            "median"
        ),
        alelos=(
            "allele",
            lambda x: ", ".join(sorted(x.unique()))
        )
    ))

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,AAVKDERAVHADM,186,198,1,3.600,HLA-DRB1*01:01
1,AAVKDERAVHADMG,186,199,1,4.100,HLA-DRB1*01:01
2,AAVKDERAVHADMGYWIE,186,203,1,4.700,HLA-DRB3*01:01
3,AAVKDERAVHADMGYWIES,186,204,1,3.300,HLA-DRB3*01:01
4,AAVKDERAVHADMGYWIESQ,186,205,1,2.300,HLA-DRB3*01:01
...,...,...,...,...,...,...
220,YRPGYHTQTAGPWHLGK,256,272,4,2.150,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*04:01/DQB1..."
221,YRPGYHTQTAGPWHLGKL,256,273,4,3.050,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*04:01/DQB1..."
222,YRPGYHTQTAGPWHLGKLE,256,274,2,1.845,"HLA-DQA1*05:01/DQB1*02:01, HLA-DQA1*05:01/DQB1..."
223,YRPGYHTQTAGPWHLGKLEL,256,275,2,2.125,"HLA-DQA1*05:01/DQB1*02:01, HLA-DQA1*05:01/DQB1..."


# Filtragem por epítopos que presentes em mais de 2 alelos.

In [17]:
epitopos_repetidos = epitopos_repetidos[
    epitopos_repetidos["qte_de_alelos"] >= 2
].reset_index(drop=True)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,ADSPKRLATAIAGAWEN,36,52,2,2.065,"HLA-DPA1*02:01/DPB1*14:01, HLA-DQA1*05:01/DQB1..."
1,ADSPKRLATAIAGAWENG,36,53,3,2.200,"HLA-DPA1*02:01/DPB1*14:01, HLA-DQA1*05:01/DQB1..."
2,ADSPKRLATAIAGAWENGV,36,54,2,1.565,"HLA-DPA1*02:01/DPB1*14:01, HLA-DQA1*05:01/DQB1..."
3,ADSPKRLATAIAGAWENGVC,36,55,2,2.095,"HLA-DPA1*02:01/DPB1*14:01, HLA-DQA1*05:01/DQB1..."
4,ADSPKRLATAIAGAWENGVCG,36,56,2,2.545,"HLA-DPA1*02:01/DPB1*14:01, HLA-DQA1*05:01/DQB1..."
...,...,...,...,...,...,...
107,YRPGYHTQTAGPWHLGK,256,272,4,2.150,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*04:01/DQB1..."
108,YRPGYHTQTAGPWHLGKL,256,273,4,3.050,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*04:01/DQB1..."
109,YRPGYHTQTAGPWHLGKLE,256,274,2,1.845,"HLA-DQA1*05:01/DQB1*02:01, HLA-DQA1*05:01/DQB1..."
110,YRPGYHTQTAGPWHLGKLEL,256,275,2,2.125,"HLA-DQA1*05:01/DQB1*02:01, HLA-DQA1*05:01/DQB1..."


In [18]:
epitopos_repetidos = (
    epitopos_repetidos
    .sort_values(
        ["median_binding_percentile", "qte_de_alelos"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,GVFTTNIWLKLRE,161,173,5,0.400,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1..."
1,HTWTEQYKFQADSPKRL,26,42,2,0.755,"HLA-DRB3*01:01, HLA-DRB5*01:01"
2,GVFTTNIWLKLREV,161,174,4,0.835,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1..."
3,QYKFQADSPKRLATAIAGA,31,49,2,1.070,"HLA-DRB3*01:01, HLA-DRB3*02:02"
4,HTWTEQYKFQADSPKRLAT,26,44,4,1.105,"HLA-DRB1*04:01, HLA-DRB3*01:01, HLA-DRB3*02:02..."
...,...,...,...,...,...,...
107,EDGCWYGMEIRPISEK,326,341,2,4.150,"HLA-DQA1*04:01/DQB1*04:02, HLA-DRB1*11:01"
108,HRLMSAAVKDERAVH,181,195,2,4.150,"HLA-DQA1*01:02/DQB1*06:02, HLA-DRB1*03:01"
109,VTNEVHTWTEQYKFQADSPK,21,40,3,4.300,"HLA-DQA1*01:01/DQB1*05:01, HLA-DRB1*08:02, HLA..."
110,LKYSWKTWGKAKIVTAETQ,111,129,2,4.400,"HLA-DQA1*05:01/DQB1*03:01, HLA-DRB1*13:02"


## Separando epítopos e criando arquivo FASTA para IEDB analysis resource

In [19]:
pepitides = epitopos_repetidos.peptide

with open("./peptideos.fasta", "w") as f:
    for i, peptide in enumerate(pepitides, start=1):
        f.write(f">NP {i}\n")
        f.write(f"{peptide}\n")
        
pepitides

0             GVFTTNIWLKLRE
1         HTWTEQYKFQADSPKRL
2            GVFTTNIWLKLREV
3       QYKFQADSPKRLATAIAGA
4       HTWTEQYKFQADSPKRLAT
               ...         
107        EDGCWYGMEIRPISEK
108         HRLMSAAVKDERAVH
109    VTNEVHTWTEQYKFQADSPK
110     LKYSWKTWGKAKIVTAETQ
111    LLWKQIANELNYILWENNIK
Name: peptide, Length: 112, dtype: str

### Seqkit remove sequências proteicas contendo gaps e *.

In [20]:
!seqkit grep -s -v -r -p '[-*X]' './Fastas/denv3_NS1_proteinas.fasta' > DENV3_seq_filter_all.fasta

### Resultado IEDB analysis resource

In [21]:
conservacy_result = pd.read_csv('./ConservancyResult_tcell_2.csv')
conservacy_result

,Epitope #,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,View details
0,1,NP 1,GVFTTNIWLKLRE,13,93.57% (597/638),84.62%,100.00%,NaN
1,2,NP 2,HTWTEQYKFQADSPKRL,17,94.67% (604/638),94.12%,100.00%,NaN
2,3,NP 3,GVFTTNIWLKLREV,14,72.88% (465/638),78.57%,100.00%,NaN
3,4,NP 4,QYKFQADSPKRLATAIAGA,19,94.04% (600/638),73.68%,100.00%,NaN
4,5,NP 5,HTWTEQYKFQADSPKRLAT,19,94.51% (603/638),84.21%,100.00%,NaN
...,...,...,...,...,...,...,...,...
107,108,NP 108,EDGCWYGMEIRPISEK,16,61.13% (390/638),87.50%,100.00%,NaN
108,109,NP 109,HRLMSAAVKDERAVH,15,80.88% (516/638),80.00%,100.00%,NaN
109,110,NP 110,VTNEVHTWTEQYKFQADSPK,20,98.75% (630/638),95.00%,100.00%,NaN
110,111,NP 111,LKYSWKTWGKAKIVTAETQ,19,94.51% (603/638),68.42%,100.00%,NaN


### Merge da coluna qte_de_alelos ao dataframe conservacy_result

In [22]:
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "qte_de_alelos"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide")).drop(columns=("View details"))

conservacy_result

,Epitope #,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos
0,1,NP 1,GVFTTNIWLKLRE,13,93.57% (597/638),84.62%,100.00%,5
1,2,NP 2,HTWTEQYKFQADSPKRL,17,94.67% (604/638),94.12%,100.00%,2
2,3,NP 3,GVFTTNIWLKLREV,14,72.88% (465/638),78.57%,100.00%,4
3,4,NP 4,QYKFQADSPKRLATAIAGA,19,94.04% (600/638),73.68%,100.00%,2
4,5,NP 5,HTWTEQYKFQADSPKRLAT,19,94.51% (603/638),84.21%,100.00%,4
...,...,...,...,...,...,...,...,...
107,108,NP 108,EDGCWYGMEIRPISEK,16,61.13% (390/638),87.50%,100.00%,2
108,109,NP 109,HRLMSAAVKDERAVH,15,80.88% (516/638),80.00%,100.00%,2
109,110,NP 110,VTNEVHTWTEQYKFQADSPK,20,98.75% (630/638),95.00%,100.00%,3
110,111,NP 111,LKYSWKTWGKAKIVTAETQ,19,94.51% (603/638),68.42%,100.00%,2


### Gerando a coluna percent_match para filtrar os epitopos com percentagem de match maior que 50%

In [23]:
col = "Percent of protein sequence matches at identity <= 100%"

conservacy_result["percent_match"] = (
    conservacy_result[col]
    .astype(str)
    .str.extract(r"(\d+(?:\.\d+)?)")[0]
    .astype(float)
)

conservacy_result

,Epitope #,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,percent_match
0,1,NP 1,GVFTTNIWLKLRE,13,93.57% (597/638),84.62%,100.00%,5,93.57
1,2,NP 2,HTWTEQYKFQADSPKRL,17,94.67% (604/638),94.12%,100.00%,2,94.67
2,3,NP 3,GVFTTNIWLKLREV,14,72.88% (465/638),78.57%,100.00%,4,72.88
3,4,NP 4,QYKFQADSPKRLATAIAGA,19,94.04% (600/638),73.68%,100.00%,2,94.04
4,5,NP 5,HTWTEQYKFQADSPKRLAT,19,94.51% (603/638),84.21%,100.00%,4,94.51
...,...,...,...,...,...,...,...,...,...
107,108,NP 108,EDGCWYGMEIRPISEK,16,61.13% (390/638),87.50%,100.00%,2,61.13
108,109,NP 109,HRLMSAAVKDERAVH,15,80.88% (516/638),80.00%,100.00%,2,80.88
109,110,NP 110,VTNEVHTWTEQYKFQADSPK,20,98.75% (630/638),95.00%,100.00%,3,98.75
110,111,NP 111,LKYSWKTWGKAKIVTAETQ,19,94.51% (603/638),68.42%,100.00%,2,94.51


### Sort por percent_match e filtragem por percent_match >= 95.00

In [24]:
conservacy_result_filtered = (
    conservacy_result[conservacy_result["percent_match"] >= 95.0]
    .sort_values(
            by="percent_match", 
            ascending=False
        ).reset_index(drop=True)
    )

conservacy_result_filtered

,Epitope #,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,percent_match
0,60,NP 60,ERAVHADMGYWIE,13,99.06% (632/638),84.62%,100.00%,2,99.06
1,65,NP 65,ERAVHADMGYWIES,14,99.06% (632/638),85.71%,100.00%,2,99.06
2,72,NP 72,ERAVHADMGYWIESQ,15,99.06% (632/638),80.00%,100.00%,2,99.06
3,57,NP 57,HTWTEQYKFQADSP,14,98.90% (631/638),92.86%,100.00%,2,98.90
4,99,NP 99,VTNEVHTWTEQYKFQA,16,98.90% (631/638),93.75%,100.00%,2,98.90
5,39,NP 39,HTWTEQYKFQADS,13,98.90% (631/638),92.31%,100.00%,2,98.90
6,89,NP 89,VTNEVHTWTEQYKFQADSP,19,98.75% (630/638),94.74%,100.00%,2,98.75
7,110,NP 110,VTNEVHTWTEQYKFQADSPK,20,98.75% (630/638),95.00%,100.00%,3,98.75
8,38,NP 38,SSFIIDGPNTPEC,13,97.96% (625/638),84.62%,100.00%,3,97.96
9,68,NP 68,SSFIIDGPNTPECP,14,97.65% (623/638),85.71%,100.00%,3,97.65
